# Predicción de Escalada de Conflicto usando XGBoost 🚀

Este notebook implementa el algoritmo **XGBoost (Extreme Gradient Boosting)** para predecir el nivel de escalada del conflicto (0 = Baja, 1 = Media, 2 = Alta).
Dado que ya procesamos los datos rigurosamente en el notebook anterior, aquí nos enfocaremos enteramente en la arquitectura del modelo, la validación correcta y la evaluación de resultados.

### Consideraciones Clave:
1. **TimeSeriesSplit:** Al ser datos históricos, usamos una validación cruzada secuencial para evitar "ver el futuro" (Data Leakage).
2. **Desbalance de Clases:** Usaremos el cálculo automático de pesos (`sample_weight`) para penalizar fuertemente al modelo cuando falle al predecir un día de "Alta Escalada".
3. **Métrica Macro F1:** Usaremos F1-Score en lugar de Accuracy para medir el éxito real del modelo ante clases minoritarias.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modelado y Evaluación
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Configuración Visual
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.titlesize': 14})

import warnings
warnings.filterwarnings('ignore')


## 1. Carga del Dataset Procesado


In [ ]:
# Cargar los datos limpios y escalados que exportamos del EDA
try:
    df = pd.read_csv('dataset_procesado_ml.csv')
    print(f"Dataset cargado correctamente. Dimensiones: {df.shape}")
except FileNotFoundError:
    print("Error: No se encontró 'dataset_procesado_ml.csv'. Asegúrate de haber corrido el notebook de EDA antes.")

# Separar Features (X) y Target (y)
target_col = 'target'
X = df.drop(columns=[target_col])
y = df[target_col]

# Verificar desbalance
display(y.value_counts(normalize=True) * 100)


## 2. Validación Cruzada Temporal (TimeSeriesSplit)

Dado que las filas mantienen un orden cronológico implícito, un `KFold` aleatorio mezclaría días del 2026 para predecir el 2024. Usamos `TimeSeriesSplit` para entrenar iterativamente expandiendo el horizonte temporal hacia el futuro.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
# Validemos visualmente los cortes
for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    print(f"Fold {fold+1}:")
    print(f"  Entrenamiento: {len(train_index)} filas | Prueba: {len(test_index)} filas")


## 3. Configuración y Entrenamiento del Modelo XGBoost

Configuramos XGBoost para clasificación multiclase (`multi:softprob`).
Crucial: Añadimos `sample_weight` durante el entrenamiento (`fit()`) para obligar al algoritmo a prestar máxima atención a la clase 2 (Alta Escalada) que es tan escasa.


In [ ]:
# Definición del modelo
# max_depth=4 (para no sobreajustar), learning_rate bajo y muchos estimadores
xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    max_depth=4,
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

fold_f1_scores = []

print("=== INICIANDO ENTRENAMIENTO CROSS-VALIDATION ===")
for fold, (train_idx, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_index]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_index]
    
    # Calcular pesos de clase dinámicamente para contrarrestar el desbalance
    weights = compute_sample_weight(class_weight='balanced', y=y_train)
    
    # Entrenar modelo
    xgb_model.fit(X_train, y_train, sample_weight=weights)
    
    # Predecir
    y_pred = xgb_model.predict(X_test)
    
    # Evaluar con Macro F1-Score (importancia igual a todas las clases)
    score = f1_score(y_test, y_pred, average='macro')
    fold_f1_scores.append(score)
    print(f"Fold {fold+1} - Macro F1-Score: {score:.4f}")

print(f"\n--> MACRO F1-SCORE PROMEDIO: {np.mean(fold_f1_scores):.4f} (+/- {np.std(fold_f1_scores):.4f})")


## 4. Evaluación Final (Matriz de Confusión)

Entrenamos un modelo final con la **totalidad de los datos** (ideal si esto se pondrá en producción a futuro) y vemos su rendimiento en el mismo set. 
*(Nota metodológica: en la vida real se separaría un set de test intocable al final de todo el periodo, pero usaremos el reporte sobre todo el dataset solo para entender qué aprendió el modelo).*


In [ ]:
# Modelo Final en todo el dataset
weights_final = compute_sample_weight(class_weight='balanced', y=y)
xgb_model.fit(X, y, sample_weight=weights_final)
y_pred_final = xgb_model.predict(X)

print("\n=== REPORTE DE CLASIFICACIÓN FINAL ===")
print(classification_report(y, y_pred_final))

# Matriz de Confusión
cm = confusion_matrix(y, y_pred_final)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baja (0)', 'Media (1)', 'Alta (2)'],
            yticklabels=['Baja (0)', 'Media (1)', 'Alta (2)'])
plt.title('Matriz de Confusión - XGBoost', pad=15)
plt.ylabel('Valor Real (Verdad)')
plt.xlabel('Predicción del Modelo')
plt.show()


## 5. Importancia de Variables (Feature Importance)

El verdadero poder de XGBoost no es solo predecir, sino decirnos **POR QUÉ** predice. Aquí extraemos cuáles son los features que dirigen la guerra según las matemáticas.


In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 8))
# Mostramos el top 20
top_20 = importances.head(20)
sns.barplot(x=top_20.values, y=top_20.index, palette='magma')
plt.title('Top 20 Features más Importantes (XGBoost Feature Importance)')
plt.xlabel('Importancia (Ganancia de Información Relativa)')
plt.ylabel('Feature')
plt.show()


### Conclusión del Modelado
1. **Desempeño:** Verifica el `Macro avg f1-score` en el reporte. Si está por encima de 0.60 - 0.70 en un contexto de guerra altamente impredecible, es un resultado muy sólido.
2. **Matriz de Confusión:** Presta especial atención a la esquina inferior derecha. ¿Logró el modelo predecir correctamente la mayoría de los días de "Alta Escalada (2)"? Gracias al balanceo de pesos, debería ser mucho mejor que un modelo tradicional.
3. **Insights Geopolíticos:** Analiza el gráfico de Importancia. ¿Fueron los Embeddings (variables de NLP) determinantes? ¿O el precio del Brent dominó las decisiones del algoritmo? Esto responde a tu tesis.
